<div class="blog-language-switch" role="group" aria-label="Article language"><span aria-current="page">English</span><a href="/ipynb/zh-CN/Computer-Science/Computer-Organization/09-virtual-memory-and-translation.html" lang="zh-CN" hreflang="zh-CN">中文</a></div>

[Back to Computer Organization and Architecture guideline](Computer-Organization.html)

## **Virtual Memory and Address Translation** {#virtual-memory-and-address-translation}

Chapter 08 followed a physical address through the cache hierarchy. Real programs usually do not generate that physical address directly. An instruction fetch, load, or store first produces a **virtual address**. The memory-management unit (MMU) translates it, verifies that the requested operation is permitted, and only then allows the cache and memory system to serve the request.

Virtual memory is therefore more than a way to use storage when RAM is full. It is an architectural contract jointly implemented by the processor and operating system. Hardware performs fast translation and permission checks; the operating system creates page tables, decides which mappings exist, and handles exceptional cases. This chapter keeps the focus on that hardware-software boundary.

### **Why Virtual Memory Exists** {#why-virtual-memory-exists}

Without virtual memory, a program would need to know where its bytes physically reside. Loading several programs would require finding non-overlapping physical ranges, moving a program could invalidate every stored pointer, and one program could accidentally name another program's data. Physical fragmentation and changing memory demand would leak into ordinary application logic.

Virtual memory inserts a level of indirection. A process works with a stable, mostly contiguous address space, while page-table mappings place its pages in any available physical frames. This indirection supports four closely related goals:

- **Isolation:** private mappings prevent one process from accessing another process's frames.
- **Relocation:** virtual addresses remain unchanged when physical placement changes.
- **Demand allocation:** unused or inactive virtual pages need not occupy RAM immediately.
- **Controlled sharing:** different processes can map the same frame with selected permissions.

::: {.diagram-scroll .wide-diagram}
![Two processes see simple virtual spaces while their code, data, and shared pages map to scattered physical frames.](assets/virtual-memory-benefits.svg){fig-align="center"}
:::

A large virtual address space is not a promise of unlimited usable memory. A page still needs physical storage when it is actively used, and page tables, RAM capacity, backing storage, and operating-system policy impose real limits. The benefit is that allocation and protection are managed in page-sized units instead of being exposed as fixed physical addresses.

<details>
<summary>Python model: the same virtual address can mean different physical bytes</summary>

```python
PAGE_SIZE = 4096

# Each process owns a different virtual-page mapping.
page_tables = {
    'A': {0x1: 12, 0x2: 27},
    'B': {0x1: 52, 0x2: 27},  # VPN 0x2 deliberately shares PFN 27
}


def translate(process, virtual_address):
    vpn, offset = divmod(virtual_address, PAGE_SIZE)
    if vpn not in page_tables[process]:
        raise MemoryError('virtual page is not mapped')
    pfn = page_tables[process][vpn]
    return pfn * PAGE_SIZE + offset


same_va = 0x1008
assert translate('A', same_va) != translate('B', same_va)

shared_va = 0x2008
assert translate('A', shared_va) == translate('B', shared_va)
print(hex(translate('A', same_va)), hex(translate('B', same_va)))
```

</details>

### **Virtual and Physical Address Spaces** {#virtual-and-physical-address-spaces}

A **virtual address space** is the set of addresses interpreted under one translation context. For a user process, that context is usually identified by a page-table root plus an address-space identifier such as an ASID or PCID. A **physical address space** is the set of addresses accepted by the memory hierarchy and memory controller. The two spaces can have different widths and different layouts.

The numeric virtual address alone is not a globally unique memory location. Process A and process B can both use virtual address `0x12345ABC`, yet their page tables may translate it to different physical frames. Conversely, two unrelated virtual addresses can intentionally map to the same physical frame for a shared library, shared-memory region, or memory-mapped file.

::: {.diagram-scroll .wide-diagram}
![The same virtual page in two address spaces maps to different frames, while two other virtual pages share one physical frame.](assets/virtual-physical-spaces.svg){fig-align="center"}
:::

A context switch changes the active translation context. A simple processor may flush process-specific TLB entries when the page-table root changes. A processor with ASID-tagged TLB entries can retain translations from several processes and distinguish them using `(ASID, VPN)` rather than VPN alone. This avoids unnecessary refill work while preserving isolation.

| Property | Virtual address space | Physical address space |
|---|---|---|
| interpreted by | process and MMU translation context | cache hierarchy, interconnect, and memory controller |
| layout | private, sparse, and selected by page tables | finite installed/architecturally addressable storage |
| same number in two processes | may identify different data | identifies the same physical location |
| holes | unmapped ranges are common | unimplemented or reserved ranges are platform-specific |
| sharing | several virtual mappings may alias one frame | one frame stores the shared bytes |

### **Pages, Frames, and Address Translation** {#pages-frames-and-address-translation}

Paging divides a virtual space into fixed-size **pages** and physical memory into equally sized **frames**. Both are aligned to the page size. Translation replaces the virtual page number (VPN) with a physical page or frame number (PPN/PFN), while preserving the byte offset inside the page.

For page size $P$ bytes and virtual address $VA$,

$$
VPN=\left\lfloor\frac{VA}{P}\right\rfloor,\qquad offset=VA\bmod P.
$$

If the selected translation maps that VPN to physical page number $PPN$, then

$$
PA=PPN\times P+offset.
$$

Here $VA$ is the virtual byte address, $PA$ is the physical byte address, $P$ is the number of bytes in one page/frame, and `offset` identifies the same byte position before and after translation. Alignment is what makes the offset reusable.

When $P=2^p$, the lowest $p$ address bits form the offset. A 4 KiB page has $P=4096=2^{12}$ bytes, so a 32-bit virtual address contains a 20-bit VPN and a 12-bit page offset.

::: {.diagram-scroll .wide-diagram}
![Virtual address 0x12345ABC is split into VPN 0x12345 and offset 0xABC; mapping the VPN to PPN 0x2BEEF creates physical address 0x2BEEFABC.](assets/page-frame-address-translation.svg){fig-align="center"}
:::

In the running example, `0x12345ABC` becomes VPN `0x12345` and offset `0xABC`. The mapping `0x12345 -> 0x2BEEF` produces physical address `0x2BEEFABC`. No addition is performed between page number and offset fields in hardware; concatenation is equivalent because both frame bases and pages are aligned.

<details>
<summary>Python example: split a virtual address and construct its physical address</summary>

```python
def split_address(address, page_size):
    # divmod returns both the page number and byte offset.
    return divmod(address, page_size)


def make_physical_address(ppn, offset, page_size):
    if not 0 <= offset < page_size:
        raise ValueError('offset must remain inside one page')
    return ppn * page_size + offset


virtual_address = 0x12345ABC
page_size = 4 * 1024
vpn, offset = split_address(virtual_address, page_size)
physical_address = make_physical_address(0x2BEEF, offset, page_size)

assert vpn == 0x12345
assert offset == 0xABC
assert physical_address == 0x2BEEFABC
print(hex(vpn), hex(offset), hex(physical_address))
```

</details>

Page size creates a trade-off. Larger pages increase TLB coverage and reduce the number of page-table entries, but may waste unused bytes inside the last page of an allocation, move more data on a fault, and require larger aligned physical regions. Many architectures therefore support a normal page size plus one or more huge-page sizes.

### **Page Tables** {#page-tables}

A **page table** is the memory-resident data structure that describes a virtual address space. A privileged register identifies its root. Each indexed entry either points to the next table level, supplies a final physical page number, or states that the requested mapping cannot currently be used.

Page-table walks are part of the memory system: their entries reside in memory and their reads pass through caches. A four-level walk does not necessarily generate four DRAM accesses, because page-walk caches and ordinary data caches may contain upper-level entries. It is nevertheless a chain of dependent lookups, which is why processors cache completed translations in a TLB.

#### **Single-Level Page Tables** {#single-level-page-tables}

A single-level page table treats the VPN as one array index. If a virtual address has $v$ bits and the page offset has $p$ bits, the table needs

$$
N_{PTE}=2^{v-p}
$$

entries. With $E$ bytes per entry, its memory cost is

$$
M_{flat}=2^{v-p}\times E.
$$

$N_{PTE}$ is the number of possible virtual pages, $v-p$ is the number of VPN bits, $E$ is PTE size, and $M_{flat}$ is the memory reserved for the flat table. For 32-bit virtual addresses, 4 KiB pages, and 4-byte PTEs, the result is $2^{20}\times4=4$ MiB per process, even when the process maps only a few pages.

<details>
<summary>Python example: calculate flat page-table storage</summary>

```python
def flat_page_table_bytes(virtual_bits, page_size, pte_bytes):
    # A power-of-two page size dedicates log2(page_size) bits to offset.
    offset_bits = page_size.bit_length() - 1
    if page_size != 1 << offset_bits:
        raise ValueError('page size must be a power of two')
    vpn_bits = virtual_bits - offset_bits
    return (1 << vpn_bits) * pte_bytes


size = flat_page_table_bytes(32, 4096, 4)
assert size == 4 * 1024 * 1024
print(f'{size / 1024**2:.0f} MiB per process')
```

</details>

Flat tables provide direct indexing and conceptually simple hardware, but their fixed cost is poorly matched to large sparse address spaces.

#### **Multi-Level Page Tables** {#multi-level-page-tables}

A multi-level page table divides the VPN into several indices. An upper-level entry points to a lower-level table only when some virtual address in that subtree is mapped. Large holes can therefore be represented by one invalid upper-level entry instead of thousands or millions of invalid leaf entries.

For the same 32-bit example, a two-level design can split the address as `10-bit L1 index | 10-bit L2 index | 12-bit offset`. With 4-byte entries, each table contains $2^{10}=1024$ entries and occupies exactly 4 KiB, so every table is itself one page.

::: {.diagram-scroll .wide-diagram}
![A flat table reserves 4 MiB, while a two-level table allocates one root and only the leaf tables required by populated virtual regions.](assets/single-vs-multilevel-page-tables.svg){fig-align="center"}
:::

For `VA = 0x12345ABC`, the upper index is 72, the lower index is 837, and the offset is `0xABC`. A hardware walk reads `root[72]` to find the relevant leaf table and then reads `leaf[837]` to obtain the final PPN and permissions.

<details>
<summary>Python model: walk a two-level page table</summary>

```python
PAGE_SIZE = 4096
INDEX_MASK = 0x3FF  # ten one-bits select one of 1024 entries

# Only one lower-level table is allocated in this small example.
root = {
    72: {837: {'ppn': 0x2BEEF, 'read': True, 'write': True}},
}


def walk_two_levels(virtual_address):
    l1_index = (virtual_address >> 22) & INDEX_MASK
    l2_index = (virtual_address >> 12) & INDEX_MASK
    offset = virtual_address & 0xFFF

    leaf_table = root.get(l1_index)
    if leaf_table is None:
        raise MemoryError('no lower-level table for this virtual region')
    pte = leaf_table.get(l2_index)
    if pte is None:
        raise MemoryError('virtual page is not mapped')
    return pte['ppn'] * PAGE_SIZE + offset, (l1_index, l2_index)


pa, indices = walk_two_levels(0x12345ABC)
assert indices == (72, 837)
assert pa == 0x2BEEFABC
print(indices, hex(pa))
```

</details>

The trade-off is explicit: a hierarchy saves memory when the virtual space is sparse, but a TLB miss may need several dependent PTE reads. Huge-page leaf entries can terminate a walk at an upper level, reducing both page-table depth and TLB pressure when a large aligned mapping is suitable.

#### **Page-Table Entries** {#page-table-entries}

A **page-table entry (PTE)** does not contain only a frame number. It also records whether the entry is usable, which privilege levels may use it, which operations are allowed, and often whether the page has been accessed or modified. Exact names and bit positions vary by architecture, so it is safer to reason about their roles than to assume one universal binary layout.

::: {.diagram-scroll .wide-diagram}
![A generic PTE combines a physical page number with validity, privilege, read, write, execute, accessed, dirty, and software-defined state.](assets/generic-pte-fields.svg){fig-align="center"}
:::

| Conceptual field | Purpose | Typical consequence |
|---|---|---|
| valid or present | states whether this entry can complete translation | clear may trigger a page fault or indicate an unused subtree |
| PPN/PFN | identifies the mapped physical frame or next table | combined with the offset for a leaf mapping |
| user/supervisor | restricts use by current privilege mode | user access to a supervisor-only page faults |
| read/write/execute | restricts the requested operation | supports read-only code, non-executable data, and guard pages |
| accessed/reference | records recent use | helps software choose replacement candidates |
| dirty/modified | records a successful write | tells software whether an evicted page needs write-back |
| software state | stores OS-specific metadata | may identify copy-on-write, swap location, or bookkeeping state |

A non-leaf PTE points to another page-table page; a leaf PTE describes a mapped page. Permissions must be checked on a TLB hit as well as during a page walk, so a TLB entry caches protection information together with the PPN.

<details>
<summary>Python model: apply PTE permission checks</summary>

```python
from dataclasses import dataclass


@dataclass
class PTE:
    ppn: int
    present: bool
    user: bool
    read: bool
    write: bool
    execute: bool
    accessed: bool = False
    dirty: bool = False


def authorize(pte, mode, operation):
    if not pte.present:
        return 'page-fault:not-present'
    if mode == 'user' and not pte.user:
        return 'page-fault:privilege'
    allowed = {'read': pte.read, 'write': pte.write, 'execute': pte.execute}
    if not allowed[operation]:
        return f'page-fault:{operation}-denied'

    # Successful accesses update usage state.
    pte.accessed = True
    if operation == 'write':
        pte.dirty = True
    return 'allowed'


data_page = PTE(0x2BEEF, True, True, True, True, False)
assert authorize(data_page, 'user', 'write') == 'allowed'
assert data_page.accessed and data_page.dirty
assert authorize(data_page, 'user', 'execute') == 'page-fault:execute-denied'
print(data_page)
```

</details>

### **Translation Lookaside Buffers** {#translation-lookaside-buffers}

A **translation lookaside buffer (TLB)** is a small associative cache of recently used page translations. Its key is conceptually `(ASID, VPN)` and its value contains the PPN plus permission and status information. It caches where a page is mapped; it does **not** cache the requested instruction or data bytes.

On a TLB hit, the MMU obtains the PPN without walking the page table. On a TLB miss, hardware or a privileged software handler looks up the page table. A valid leaf PTE refills the TLB and the original access continues. Only a missing, invalid, or prohibited mapping produces a page fault. Therefore, `TLB miss` and `page fault` are not synonyms.

![A simplified virtual-to-physical translation flow checks the TLB first, consults the page table on a TLB miss, and reaches backing storage only for the diagram's nonresident-page case.](assets/tlb-address-resolution-wikimedia.svg){fig-align="center"}

*Image source: [Arilou, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:TLB.svg), licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). The source diagram uses 'page table miss' for a simplified nonresident-page path; real page faults also include permission failures, lazy allocation, and copy-on-write.*

The amount of virtual memory that can be represented simultaneously is the **TLB reach**:

$$
R_{TLB}=N_{entries}\times P.
$$

$R_{TLB}$ is the reachable byte range if entries map distinct pages, $N_{entries}$ is the number of cached translations, and $P$ is page size. A 64-entry TLB with 4 KiB pages reaches 256 KiB; the same number of 2 MiB mappings reaches 128 MiB. Larger pages improve reach but retain the allocation and fragmentation trade-offs described earlier.

TLBs are commonly split into instruction and data structures near the pipeline, with a larger shared second-level TLB on some processors. Replacement, associativity, ASID capacity, page-size support, and separate read/write/execute permissions all affect behavior.

<details>
<summary>Python model: an ASID-aware LRU TLB</summary>

```python
from collections import OrderedDict


class TLB:
    def __init__(self, entries):
        self.entries = entries
        self.lines = OrderedDict()

    def lookup(self, asid, vpn):
        key = (asid, vpn)
        if key not in self.lines:
            return None
        value = self.lines.pop(key)
        self.lines[key] = value  # most recently used
        return value

    def insert(self, asid, vpn, ppn):
        key = (asid, vpn)
        self.lines.pop(key, None)
        self.lines[key] = ppn
        if len(self.lines) > self.entries:
            self.lines.popitem(last=False)  # evict least recently used


tlb = TLB(entries=2)
tlb.insert(asid=7, vpn=0x12345, ppn=0x2BEEF)
tlb.insert(asid=12, vpn=0x12345, ppn=0x01020)
assert tlb.lookup(7, 0x12345) == 0x2BEEF
assert tlb.lookup(12, 0x12345) == 0x01020
print(tlb.lines)
```

</details>

### **Page Faults and Demand Paging** {#page-faults-and-demand-paging}

A **page fault** is a precise exception raised when the current translation and protection state cannot complete a memory access. The processor records enough architectural state for the operating system to inspect the fault and either repair the mapping or report an error. If repaired, the faulting instruction is restarted and behaves as if it had simply taken a long time.

Demand paging deliberately begins with some valid virtual regions marked nonresident. The first access faults, the operating system obtains a frame, prepares its contents, updates the PTE and relevant TLB state, and restarts the instruction. Preparing the page may mean reading a file or swap area, but it may instead mean zero-filling a fresh anonymous page or copying an existing page for copy-on-write. Thus, not every page fault performs storage I/O.

::: {.diagram-scroll .wide-diagram}
![A page fault traps to the operating system, which rejects invalid accesses or prepares a frame, updates translation state, and restarts the instruction.](assets/page-fault-handling-flow.svg){fig-align="center"}
:::

| Fault situation | Mapping meaning | Typical handling |
|---|---|---|
| lazy anonymous allocation | valid region, no frame yet | allocate and zero-fill a frame |
| file-backed demand page | valid file mapping, page not resident | read the required file page |
| swapped-out page | prior contents stored outside RAM | read page from swap/storage |
| copy-on-write store | shared read-only mapping marked for COW | copy frame, install private writable PTE |
| protection violation | operation conflicts with R/W/X or privilege bits | reject access or signal the process |
| unmapped address | no valid virtual region | reject access, often as a segmentation fault |

A replacement decision may be required when no free frame exists. If the victim page is dirty, its current bytes must be preserved before the frame is reused; a clean file-backed page may simply be discarded because it can be read again. These are operating-system policies built on the accessed and dirty information exposed by the architecture.

<details>
<summary>Python model: classify and resolve simplified page faults</summary>

```python
regions = {
    0x10: {'kind': 'anonymous', 'write': True, 'present': False},
    0x20: {'kind': 'file', 'write': False, 'present': False},
    0x30: {'kind': 'cow', 'write': False, 'present': True},
}
next_free_ppn = iter([100, 101, 102])


def handle_access(vpn, operation):
    region = regions.get(vpn)
    if region is None:
        return 'fatal:unmapped'

    if operation == 'write' and not region['write']:
        if region['kind'] != 'cow':
            return 'fatal:write-protection'
        # Copy-on-write creates a private writable page.
        region.update(kind='anonymous', write=True, ppn=next(next_free_ppn))
        return 'resolved:copy-on-write'

    if not region['present']:
        region.update(present=True, ppn=next(next_free_ppn))
        return f"resolved:fill-{region['kind']}"
    return 'hit:resident'


assert handle_access(0x10, 'write') == 'resolved:fill-anonymous'
assert handle_access(0x20, 'read') == 'resolved:fill-file'
assert handle_access(0x30, 'write') == 'resolved:copy-on-write'
assert handle_access(0x99, 'read') == 'fatal:unmapped'
print(regions)
```

</details>

### **Protection, Privilege, and Sharing** {#protection-privilege-and-sharing}

Address translation provides a natural place to enforce protection because every instruction fetch, load, and store already passes through the MMU. A page can be readable but not writable, executable but not writable, accessible only in supervisor mode, or completely inaccessible as a guard region. These rules are checked before the requested cache access is allowed to affect architectural state.

Page granularity also makes sharing selective. Two processes can map the same read-only executable frame at different virtual addresses while keeping writable data private. Shared-memory pages may be writable in both processes when synchronization is intentional. Copy-on-write initially maps one frame read-only into several processes; the first writer faults and receives a private copy.

::: {.diagram-scroll .wide-diagram}
![Two processes share code and copy-on-write frames while retaining private writable data and being blocked from supervisor-only memory.](assets/protection-sharing-mappings.svg){fig-align="center"}
:::

The same physical frame can have different permissions in different virtual mappings. Permission belongs to the mapping, not inherently to the RAM cells. This supports a shared library that is executable in user mode, a kernel alias with different privilege, or a file mapping that is read-only in one process and private copy-on-write in another.

Security depends on keeping all translation caches coherent with page-table changes. After revoking access or remapping a page, privileged software must invalidate stale TLB entries on every affected processor. Otherwise, an old cached translation could continue to authorize an access after the PTE has changed.

<details>
<summary>Python model: copy-on-write separates one shared frame on the first store</summary>

```python
frames = {27: bytearray(b'shared page')}
mappings = {
    'A': {'pfn': 27, 'writable': False, 'cow': True},
    'B': {'pfn': 27, 'writable': False, 'cow': True},
}


def write_first_byte(process, value):
    mapping = mappings[process]
    if not mapping['writable']:
        if not mapping['cow']:
            raise PermissionError('read-only mapping')
        # Fault handler allocates and copies before granting write access.
        new_pfn = max(frames) + 1
        frames[new_pfn] = bytearray(frames[mapping['pfn']])
        mapping.update(pfn=new_pfn, writable=True, cow=False)
    frames[mapping['pfn']][0] = value


write_first_byte('A', ord('S'))
assert mappings['A']['pfn'] != mappings['B']['pfn']
assert frames[mappings['A']['pfn']] == bytearray(b'Shared page')
assert frames[mappings['B']['pfn']] == bytearray(b'shared page')
print(mappings)
```

</details>

### **The Complete TLB-Cache-Memory Access Path** {#the-complete-tlb-cache-memory-access-path}

The complete path joins this chapter's translation machinery to Chapter 08's cache hierarchy. For a typical physically tagged cache, the conceptual sequence is:

1. The instruction produces a virtual address and an operation type.
2. The page offset is separated from the VPN.
3. The TLB searches for `(ASID, VPN)` and checks permissions.
4. On a TLB miss, the MMU walks the page table; a valid leaf refills the TLB, while an unusable mapping raises a page fault.
5. The PPN and unchanged page offset form the physical address.
6. The cache checks the physical block; a cache miss continues through lower cache levels and possibly DRAM.

::: {.diagram-scroll .wide-diagram}
![A virtual address checks the TLB, walks page tables on a translation miss, forms a physical address, and then checks the cache hierarchy.](assets/complete-translation-cache-path.svg){fig-align="center"}
:::

This produces independent outcomes. A TLB miss can be followed by a cache hit after the walk supplies a valid physical address. A TLB hit can be followed by a cache miss because the translation is cached but the requested data block is not. A page fault stops the requested data access before it can complete, although the page-table walk itself may have generated cache traffic.

Modern L1 caches often overlap part of translation with cache indexing using a **virtually indexed, physically tagged (VIPT)** organization. Because translation preserves the page offset, index bits taken entirely from that offset are available before the PPN arrives. A useful design condition is

$$
S\times B\le P,
$$

where $S$ is the number of cache sets, $B$ is cache-block size, and $P$ is page size. The product $S\times B$ is cache capacity per way. Keeping it within one page ensures that all set-index bits lie in the unchanged page offset; the physical tag can then arrive from the TLB while the indexed set is being read. More aggressive designs need additional alias handling.

| Translation result | Cache result | Meaning |
|---|---|---|
| TLB hit | cache hit | common fast path: translation and data are both cached |
| TLB miss, valid PTE | cache hit | translation must be reconstructed, requested block is resident |
| TLB hit | cache miss | translation is available, data comes from a lower cache or DRAM |
| page fault | no completed requested lookup | software must repair or reject the mapping first |

<details>
<summary>Python model: distinguish TLB, cache, and page-fault outcomes</summary>

```python
class TranslationCachePath:
    def __init__(self, page_table, page_size=4096, block_size=64):
        self.page_table = page_table
        self.page_size = page_size
        self.block_size = block_size
        self.tlb = {}
        self.cache_blocks = set()

    def access(self, virtual_address):
        vpn, offset = divmod(virtual_address, self.page_size)
        tlb_hit = vpn in self.tlb

        if tlb_hit:
            ppn = self.tlb[vpn]
        else:
            ppn = self.page_table.get(vpn)
            if ppn is None:
                return {'page_fault': True, 'tlb_hit': False, 'cache_hit': None}
            self.tlb[vpn] = ppn  # successful page walk refills the TLB

        physical_address = ppn * self.page_size + offset
        block = physical_address // self.block_size
        cache_hit = block in self.cache_blocks
        self.cache_blocks.add(block)  # model a refill after any miss
        return {
            'page_fault': False,
            'tlb_hit': tlb_hit,
            'cache_hit': cache_hit,
            'physical_address': physical_address,
        }


system = TranslationCachePath({0x12345: 0x2BEEF, 0x22222: 0x2BEEF})
results = [
    system.access(0x12345000),  # TLB miss, cache miss
    system.access(0x12345004),  # TLB hit, same cache block hits
    system.access(0x22222000),  # TLB miss, aliased physical block hits
    system.access(0x22222080),  # TLB hit, different cache block misses
    system.access(0x33333000),  # no mapping: page fault
]

assert [(r['page_fault'], r['tlb_hit'], r['cache_hit']) for r in results] == [
    (False, False, False),
    (False, True, True),
    (False, False, True),
    (False, True, False),
    (True, False, None),
]
print(results)
```

</details>

### **Translation Performance and Memory Overheads** {#translation-performance-and-memory-overheads}

Translation affects performance in three different ways: page tables consume memory, TLB misses add ordinary access latency, and page faults introduce a much rarer but potentially enormous delay. Keeping these scales separate prevents misleading averages.

A simplified average for accesses that do not page fault is

$$
EAT_{normal}=hT_{hit}+(1-h)T_{miss}.
$$

$EAT_{normal}$ is effective time for ordinary resident accesses, $h$ is TLB hit rate, $T_{hit}$ is the complete TLB-hit path, and $T_{miss}$ is the path including page-table walk, TLB refill, and the requested cache access. This formula is deliberately abstract because each PTE read can hit at a different cache level and processors may overlap parts of the walk.

A separate model includes page-fault probability $p$:

$$
EAT=(1-p)T_{normal}+pT_{fault}.
$$

$T_{normal}$ is the resident path, while $T_{fault}$ includes trap handling and whatever work resolves the fault. If $T_{normal}=100$ ns and a storage-backed fault costs 8 ms, keeping the average below 200 ns requires approximately

$$
p\le\frac{200-100}{8{,}000{,}000-100}\approx1.25\times10^{-5}.
$$

That is fewer than roughly one such fault per 80,000 memory accesses. The precise numbers vary greatly, but the orders-of-magnitude difference explains why sustained paging can dominate execution time.

::: {.diagram-scroll .wide-diagram}
![Translation performance combines TLB coverage, weighted hit and miss paths, and the rare but extremely costly page-fault path.](assets/translation-performance-overheads.svg){fig-align="center"}
:::

Page-table memory must also be counted. A flat table pays for every possible VPN. A multi-level table pays for a root plus allocated lower-level pages. Huge pages reduce the number of leaf entries and increase TLB reach, while potentially increasing internal fragmentation and making contiguous allocation harder.

| Mechanism | Primary benefit | Main cost or limit |
|---|---|---|
| larger or more associative TLB | fewer replacement/conflict misses | more area, energy, and lookup complexity |
| ASID/PCID tagging | retains translations across context switches | finite identifier space and invalidation complexity |
| page-walk cache | avoids repeated upper-level PTE reads | does not cache all leaf translations like a TLB |
| multi-level page table | allocates metadata only for populated regions | more dependent reads on an uncached walk |
| huge pages | greater TLB reach and shorter walks | fragmentation, alignment, and promotion/demotion cost |
| demand paging | avoids loading untouched pages | first use faults and may require slow I/O |

<details>
<summary>Python example: compare TLB reach and page-fault sensitivity</summary>

```python
def tlb_reach(entries, page_bytes):
    return entries * page_bytes


def effective_access_time(normal_ns, fault_ns, fault_probability):
    return (1 - fault_probability) * normal_ns + fault_probability * fault_ns


small_page_reach = tlb_reach(64, 4 * 1024)
huge_page_reach = tlb_reach(64, 2 * 1024 * 1024)
assert small_page_reach == 256 * 1024
assert huge_page_reach == 128 * 1024 * 1024

normal_ns = 100
fault_ns = 8_000_000
target_ns = 200
max_fault_probability = (target_ns - normal_ns) / (fault_ns - normal_ns)

assert effective_access_time(normal_ns, fault_ns, max_fault_probability) == target_ns
print(f'4 KiB reach: {small_page_reach / 1024:.0f} KiB')
print(f'2 MiB reach: {huge_page_reach / 1024**2:.0f} MiB')
print(f'maximum storage-backed fault probability: {max_fault_probability:.3e}')
```

</details>

**Chapter summary.** Virtual memory gives each process a protected address space whose pages can be relocated, allocated on demand, and deliberately shared. The MMU divides a virtual address into VPN and page offset, translates the VPN through page tables or a cached TLB entry, enforces privilege and R/W/X permissions, and combines the resulting PPN with the unchanged offset. Flat page tables offer direct indexing but consume fixed space; multi-level tables exploit sparse address spaces at the cost of dependent walks. A TLB miss requests a translation lookup, a cache miss requests a data block, and a page fault transfers control to software because the current mapping cannot complete the access. Performance depends on page-table memory, TLB reach, walk latency, and the much larger cost of storage-backed faults. Chapter 10 will follow the I/O, interrupt, DMA, and storage mechanisms that let devices move data and that ultimately service some page-fault paths.